In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

#  Load Data 
train_df = pd.read_csv(r'C:\Users\gj153\Desktop\train.csv')
test_df = pd.read_csv(r'C:\Users\gj153\Desktop\test.csv')

def super_feature_engineering(df, is_train=True):
    """
    Super Feature Engineering - Fixed Version (Handles NaN issues)
    """
    df = df.copy()
    
    #  1. Basic Feature Splitting 
    df[['Deck', 'Num', 'Side']] = df['Cabin'].str.split('/', expand=True)
    df['Num'] = pd.to_numeric(df['Num'], errors='coerce')
    
    df[['GroupId', 'PersonId']] = df['PassengerId'].str.split('_', expand=True)
    df['GroupId'] = pd.to_numeric(df['GroupId'], errors='coerce')
    df['PersonId'] = pd.to_numeric(df['PersonId'], errors='coerce')
    
    #  2. Spending Features (Crucial) 
    amenity_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    
    # Base spending features
    df['TotalSpend'] = df[amenity_cols].sum(axis=1)
    df['HasSpent'] = (df['TotalSpend'] > 0).astype(int)
    df['NumAmenities'] = (df[amenity_cols] > 0).sum(axis=1)
    df['AvgSpend'] = df['TotalSpend'] / (df['NumAmenities'] + 1)
    df['MaxSpend'] = df[amenity_cols].max(axis=1)
    df['MinSpend'] = df[amenity_cols].min(axis=1)
    
    # Spending distribution features
    df['StdSpend'] = df[amenity_cols].std(axis=1)
    df['StdSpend'].fillna(0, inplace=True)
    
    df['MedianSpend'] = df[amenity_cols].median(axis=1)
    df['RangeSpend'] = df['MaxSpend'] - df['MinSpend']
    
    # Spend concentration (Max spend ratio)
    df['SpendConcentration'] = df['MaxSpend'] / (df['TotalSpend'] + 1)
    
    # Ratio for each amenity type
    for col in amenity_cols:
        df[f'{col}_ratio'] = df[col] / (df['TotalSpend'] + 1)
    
    # Spend level (Fix: Convert to codes first, then handle NaN)
    spend_categories = pd.cut(df['TotalSpend'], 
                               bins=[-1, 0, 1000, 5000, 10000, np.inf],
                               labels=[0, 1, 2, 3, 4])
    df['SpendLevel'] = spend_categories.cat.codes.astype(int)
    df['SpendLevel'] = df['SpendLevel'].replace(-1, 0)  # Convert NaN to 0
    
    # High/Low spender indicators
    df['IsHighSpender'] = (df['TotalSpend'] > df['TotalSpend'].quantile(0.75)).astype(int)
    df['IsLowSpender'] = (df['TotalSpend'] < df['TotalSpend'].quantile(0.25)).astype(int)
    
    #  3. CryoSleep Features (Key) 
    df['CryoSleep'].fillna(False, inplace=True)
    df['CryoSleep_int'] = df['CryoSleep'].astype(int)
    
    # Mismatch between CryoSleep status and spending status
    df['CryoSleepSpendMismatch'] = df['CryoSleep_int'] * df['HasSpent']
    
    #  4. VIP Features 
    df['VIP'].fillna(False, inplace=True)
    df['VIP_int'] = df['VIP'].astype(int)
    
    # VIP status and spending interaction
    df['VIPSpendInteraction'] = df['VIP_int'] * df['TotalSpend']
    
    #  5. Cabin Features 
    # Deck encoding
    deck_map = {'A': 7, 'B': 6, 'C': 5, 'D': 4, 'E': 3, 'F': 2, 'G': 1, 'Unknown': 0}
    df['DeckNum'] = df['Deck'].map(deck_map)
    df['DeckNum'].fillna(0, inplace=True)
    
    # Side encoding
    df['SideNum'] = (df['Side'] == 'S').astype(int)
    
    # Cabin number features
    df['CabinNum'] = df['Num']
    df['CabinNum'].fillna(df['Num'].median(), inplace=True)
    df['CabinNum_log'] = np.log1p(df['CabinNum'])
    
    # High/Low deck indicators
    df['IsHighDeck'] = df['Deck'].isin(['A', 'B', 'C']).astype(int)
    df['IsLowDeck'] = df['Deck'].isin(['F', 'G']).astype(int)
    
    # Cabin placement and spending interactions
    df['DeckSpendInteraction'] = df['DeckNum'] * df['TotalSpend']
    df['HighDeckSpendRatio'] = df['IsHighDeck'] * df['HasSpent']
    
    #  6. Age Features 
    # Fill missing Age values with the group median based on HomePlanet
    df['Age'] = df.groupby('HomePlanet')['Age'].transform(
        lambda x: x.fillna(x.median())
    )
    df['Age'].fillna(df['Age'].median(), inplace=True)
    
    # Age group segmentation (Fix: Use cat.codes to handle NaN)
    age_categories = pd.cut(df['Age'], 
                             bins=[0, 12, 18, 35, 60, 100],
                             labels=[0, 1, 2, 3, 4])
    df['AgeGroup'] = age_categories.cat.codes.astype(int)
    df['AgeGroup'] = df['AgeGroup'].replace(-1, 0)  # Convert NaN to 0
    
    # Age life-stage indicators
    df['IsChild'] = (df['Age'] < 13).astype(int)
    df['IsTeenager'] = ((df['Age'] >= 13) & (df['Age'] < 18)).astype(int)
    df['IsAdult'] = ((df['Age'] >= 18) & (df['Age'] < 60)).astype(int)
    df['IsElderly'] = (df['Age'] >= 60).astype(int)
    
    # Age and spending interaction
    df['AgeSpendInteraction'] = df['Age'] * df['TotalSpend']
    df['AgeLog'] = np.log1p(df['Age'])
    
    #  7. Group Features (Important) 
    # Group size calculation
    group_size = df.groupby('GroupId').size().reset_index(name='GroupSize')
    df = df.merge(group_size, on='GroupId', how='left')
    
    # Travel party type indicators
    df['IsSolo'] = (df['GroupSize'] == 1).astype(int)
    df['IsLargeGroup'] = (df['GroupSize'] > 4).astype(int)
    
    # Group level aggregated spending statistics
    group_spend = df.groupby('GroupId')['TotalSpend'].agg(
        ['sum', 'mean', 'std', 'min', 'max']
    ).reset_index()
    group_spend.columns = ['GroupId', 'GroupTotalSpend', 'GroupAvgSpend', 
                           'GroupStdSpend', 'GroupMinSpend', 'GroupMaxSpend']
    df = df.merge(group_spend, on='GroupId', how='left')
    df['GroupStdSpend'].fillna(0, inplace=True)
    
    # Individual spend share within the group
    df['PersonSpendRatio'] = df['TotalSpend'] / (df['GroupTotalSpend'] + 1)
    
    # Group spending tier categorization (Fix: Use cat.codes)
    group_spend_categories = pd.cut(df['GroupTotalSpend'],
                                     bins=[-1, 0, 2000, 10000, 50000, np.inf],
                                     labels=[0, 1, 2, 3, 4])
    df['GroupSpendLevel'] = group_spend_categories.cat.codes.astype(int)
    df['GroupSpendLevel'] = df['GroupSpendLevel'].replace(-1, 0)
    
    # Presence of a high spender within the group
    df['GroupHasHighSpender'] = (df['GroupTotalSpend'] > df['GroupTotalSpend'].quantile(0.75)).astype(int)
    
    # Group size and spending interaction
    df['GroupSizeSpendInteraction'] = df['GroupSize'] * df['TotalSpend']
    
    # Spending variance profile within the group
    df['GroupSpendVariance'] = df['GroupStdSpend'] / (df['GroupAvgSpend'] + 1)
    
    #  8. Planetary Features 
    # HomePlanet label mapping
    planet_map = {'Earth': 0, 'Europa': 1, 'Mars': 2}
    df['HomePlanetNum'] = df['HomePlanet'].map(planet_map)
    df['HomePlanet'].fillna('Earth', inplace=True)
    df['HomePlanetNum'].fillna(0, inplace=True)
    
    # Destination mode fallback mapping
    df['Destination'].fillna(df['Destination'].mode()[0], inplace=True)
    dest_map = {'TRAPPIST-1e': 0, '55 Cancri e': 1, 'PSO J318.5-22': 2}
    df['DestinationNum'] = df['Destination'].map(dest_map)
    df['DestinationNum'].fillna(0, inplace=True)
    
    # Match criteria for same origin/destination planet
    df['SamePlanetDest'] = (df['HomePlanet'] == df['Destination']).astype(int)
    
    # Planet profile and spending interaction
    df['PlanetSpendInteraction'] = df['HomePlanetNum'] * df['TotalSpend']
    
    #  9. Mathematical Non-linear Transformations 
    # Polynomial features for TotalSpend
    df['TotalSpend_squared'] = df['TotalSpend'] ** 2
    df['TotalSpend_log'] = np.log1p(df['TotalSpend'])
    df['TotalSpend_sqrt'] = np.sqrt(df['TotalSpend'])
    
    # Polynomial features for Age
    df['Age_squared'] = df['Age'] ** 2
    df['Age_log'] = np.log1p(df['Age'])
    
    # Log transformation for GroupSize
    df['GroupSize_log'] = np.log1p(df['GroupSize'])
    
    #  10. Multi-variable Interaction Terms 
    df['HasSpentAndVIP'] = df['HasSpent'] * df['VIP_int']
    df['HasSpentAndHighDeck'] = df['HasSpent'] * df['IsHighDeck']
    df['CryoSleepAndVIP'] = df['CryoSleep_int'] * df['VIP_int']
    df['SoloAndHighSpender'] = df['IsSolo'] * df['IsHighSpender']
    
    #  11. Final Missing Value Imputation 
    for col in amenity_cols:
        df[col].fillna(0, inplace=True)
    
    #  12. Drop Redundant Columns 
    df = df.drop(['PassengerId', 'Name', 'Cabin', 'Deck', 'Side', 'Num',
                  'HomePlanet', 'Destination'], axis=1)
    
    return df

# Execute Super Feature Engineering
print("=" * 70)
print("Applying Super Feature Engineering Pipeline")
print("=" * 70)

train_processed = super_feature_engineering(train_df)
test_processed = super_feature_engineering(test_df)

print(f"✓ Feature Count: {train_processed.shape[1]}")
print(f"✓ Missing Values Remaining: {train_processed.isnull().sum().sum()}")

# Display New Feature List
print(f"\nGenerated Feature List:")
for i, col in enumerate(train_processed.columns, 1):
    print(f"  {i:2d}. {col}")

Applying Super Feature Engineering Pipeline
✓ Feature Count: 75
✓ Missing Values Remaining: 943

Generated Feature List:
   1. CryoSleep
   2. Age
   3. VIP
   4. RoomService
   5. FoodCourt
   6. ShoppingMall
   7. Spa
   8. VRDeck
   9. Transported
  10. GroupId
  11. PersonId
  12. TotalSpend
  13. HasSpent
  14. NumAmenities
  15. AvgSpend
  16. MaxSpend
  17. MinSpend
  18. StdSpend
  19. MedianSpend
  20. RangeSpend
  21. SpendConcentration
  22. RoomService_ratio
  23. FoodCourt_ratio
  24. ShoppingMall_ratio
  25. Spa_ratio
  26. VRDeck_ratio
  27. SpendLevel
  28. IsHighSpender
  29. IsLowSpender
  30. CryoSleep_int
  31. CryoSleepSpendMismatch
  32. VIP_int
  33. VIPSpendInteraction
  34. DeckNum
  35. SideNum
  36. CabinNum
  37. CabinNum_log
  38. IsHighDeck
  39. IsLowDeck
  40. DeckSpendInteraction
  41. HighDeckSpendRatio
  42. AgeGroup
  43. IsChild
  44. IsTeenager
  45. IsAdult
  46. IsElderly
  47. AgeSpendInteraction
  48. AgeLog
  49. GroupSize
  50. IsSolo
  51. I

In [2]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score
import lightgbm as lgb

#  Feature Preparation 
print("\n" + "=" * 70)
print("Feature and Matrix Alignment")
print("=" * 70)

X_train = train_processed.drop('Transported', axis=1)
y_train = train_processed['Transported'].astype(int)
X_test = test_processed.copy()

# Categorical Label Encoding
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()
label_encoders = {}

for col in categorical_features:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))
    label_encoders[col] = le

X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

print(f"✓ Total Estimator Features: {X_train.shape[1]}")
print(f"✓ Training Matrix Rows: {X_train.shape[0]}")
print(f"✓ Evaluation Matrix Rows: {X_test.shape[0]}")

#  Advanced Hyperparameter Tuning 
print("\n" + "=" * 70)
print("Advanced Hyperparameter Search (Bayesian-like Stochastic Search)")
print("=" * 70)

# Set Parameter Optimization Range Bounds
param_search_space = {
    'num_leaves': [50, 60, 70, 80, 90],
    'max_depth': [7, 8, 9, 10],
    'learning_rate': [0.03, 0.04, 0.05, 0.06, 0.07],
    'min_child_samples': [15, 20, 25, 30],
    'subsample': [0.75, 0.8, 0.85, 0.9],
    'colsample_bytree': [0.75, 0.8, 0.85, 0.9],
    'reg_alpha': [0.01, 0.05, 0.1, 0.2],
    'reg_lambda': [0.01, 0.05, 0.1, 0.2],
}

# Run Stochastic Space Evaluation (Highly Efficient)
np.random.seed(42)
best_score = 0
best_params = None
results = []

n_trials = 150  # Evaluating 150 hyperparameter space points

print(f"Testing {n_trials} Parameter Combinations\n")

for trial in range(n_trials):
    # Draw parameters stochastically from predefined spaces
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'boosting_type': 'gbdt',
        'verbose': -1,
        'random_state': 42,
        'num_threads': -1,
        'num_leaves': int(np.random.choice(param_search_space['num_leaves'])),
        'max_depth': int(np.random.choice(param_search_space['max_depth'])),
        'learning_rate': float(np.random.choice(param_search_space['learning_rate'])),
        'min_child_samples': int(np.random.choice(param_search_space['min_child_samples'])),
        'subsample': float(np.random.choice(param_search_space['subsample'])),
        'colsample_bytree': float(np.random.choice(param_search_space['colsample_bytree'])),
        'reg_alpha': float(np.random.choice(param_search_space['reg_alpha'])),
        'reg_lambda': float(np.random.choice(param_search_space['reg_lambda'])),
    }
    
    # 3-Fold Stratified Cross Validation
    cv_scores = []
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]
        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]
        
        train_fold_data = lgb.Dataset(X_fold_train, label=y_fold_train)
        val_fold_data = lgb.Dataset(X_fold_val, label=y_fold_val, reference=train_fold_data)
        
        model = lgb.train(
            params,
            train_fold_data,
            num_boost_round=500,
            valid_sets=[val_fold_data],
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
        )
        
        y_pred = model.predict(X_fold_val)
        score = roc_auc_score(y_fold_val, y_pred)
        cv_scores.append(score)
    
    mean_score = np.mean(cv_scores)
    
    results.append({
        'trial': trial + 1,
        'params': params,
        'score': mean_score
    })
    
    if mean_score > best_score:
        best_score = mean_score
        best_params = params
    
    if (trial + 1) % 30 == 0:
        print(f"[{trial+1}/{n_trials}] Current Best Cross-Validated ROC-AUC: {best_score:.6f}")

print(f"\n✓ Global Optimal Cross-Validated ROC-AUC: {best_score:.6f}")
print(f"\nOptimal Parameter Configuration Parameters:")
for key, value in best_params.items():
    if key not in ['objective', 'metric', 'boosting_type', 'verbose', 'random_state', 'num_threads']:
        print(f"  {key}: {value}")


Feature and Matrix Alignment
✓ Total Estimator Features: 74
✓ Training Matrix Rows: 8693
✓ Evaluation Matrix Rows: 4277

Advanced Hyperparameter Search (Bayesian-like Stochastic Search)
Testing 150 Parameter Combinations

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[145]	valid_0's binary_logloss: 0.380188
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[168]	valid_0's binary_logloss: 0.396158
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[126]	valid_0's binary_logloss: 0.387639
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[137]	valid_0's binary_logloss: 0.380814
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[116]	valid_0's binary_logloss: 0.396798
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration 

In [3]:
from sklearn.metrics import roc_curve, f1_score, precision_recall_curve

print("\n" + "=" * 70)
print("Classification Decision Threshold Optimization")
print("=" * 70)

# Evaluate validation split to identify best operational boundary
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

train_data = lgb.Dataset(X_tr, label=y_tr)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

# Fit temporary threshold model with optimized configurations
threshold_model = lgb.train(
    best_params,
    train_data,
    num_boost_round=600,
    valid_sets=[val_data],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

y_pred_proba = threshold_model.predict(X_val)

# Scan potential operational boundaries
thresholds = np.arange(0.35, 0.65, 0.01)
f1_scores = []
accuracies = []

for threshold in thresholds:
    y_pred = (y_pred_proba > threshold).astype(int)
    f1 = f1_score(y_val, y_pred)
    acc = accuracy_score(y_val, y_pred)
    f1_scores.append(f1)
    accuracies.append(acc)

best_threshold_f1 = thresholds[np.argmax(f1_scores)]
best_threshold_acc = thresholds[np.argmax(accuracies)]

print(f"Optimal Threshold Optimized for F1 Score: {best_threshold_f1:.4f} (F1 Max={np.max(f1_scores):.4f})")
print(f"Optimal Threshold Optimized for Overall Accuracy: {best_threshold_acc:.4f} (Acc Max={np.max(accuracies):.4f})")

# Set F1 score optimal boundary as global operational cutpoint
optimal_threshold = best_threshold_f1
print(f"\n✓ Selected Decision Threshold Cutpoint: {optimal_threshold:.4f}")

# Compare performance metrics against standard benchmarks
y_pred_optimal = (y_pred_proba > optimal_threshold).astype(int)
y_pred_default = (y_pred_proba > 0.5).astype(int)

print(f"\nOperational Boundary Performance Metrics Comparison:")
print(f"  Standard Baseline Cutpoint (0.5000) Accuracy: {accuracy_score(y_val, y_pred_default):.4f}")
print(f"  Optimized Decision Cutpoint ({optimal_threshold:.4f}) Accuracy: {accuracy_score(y_val, y_pred_optimal):.4f}")
print(f"  Net Performance Improvement Delta: {(accuracy_score(y_val, y_pred_optimal) - accuracy_score(y_val, y_pred_default)):.4f}")


Classification Decision Threshold Optimization
Training until validation scores don't improve for 50 rounds
[100]	valid_0's binary_logloss: 0.367968
Early stopping, best iteration is:
[104]	valid_0's binary_logloss: 0.367919
Optimal Threshold Optimized for F1 Score: 0.3700 (F1 Max=0.8332)
Optimal Threshold Optimized for Overall Accuracy: 0.5600 (Acc Max=0.8252)

✓ Selected Decision Threshold Cutpoint: 0.3700

Operational Boundary Performance Metrics Comparison:
  Standard Baseline Cutpoint (0.5000) Accuracy: 0.8166
  Optimized Decision Cutpoint (0.3700) Accuracy: 0.8200
  Net Performance Improvement Delta: 0.0035


In [4]:
print("\n" + "=" * 70)
print("Data Cleansing & Outlier Engineering Optimization")
print("=" * 70)

# Define Interquartile Range outlier detection logic
def detect_outliers_iqr(data, column):
    """Detect outliers using the Interquartile Range (IQR) method"""
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return (data[column] < lower_bound) | (data[column] > upper_bound)

# Engineer outlier indication masks
outlier_features = ['TotalSpend', 'Age', 'GroupSize', 'CabinNum']
for col in outlier_features:
    if col in X_train.columns:
        X_train[f'{col}_is_outlier'] = detect_outliers_iqr(X_train, col).astype(int)
        X_test[f'{col}_is_outlier'] = detect_outliers_iqr(X_test, col).astype(int)

print(f"✓ Outlier indication tracking matrices generated")
print(f"✓ Adjusted Model Matrix Features: {X_train.shape[1]}")


Data Cleansing & Outlier Engineering Optimization
✓ Outlier indication tracking matrices generated
✓ Adjusted Model Matrix Features: 78


In [5]:
print("\n" + "=" * 70)
print("Final Full Model Training Production Run")
print("=" * 70)

# Re-fit production model on complete training database
train_data_full = lgb.Dataset(X_train, label=y_train)

final_params = best_params.copy()

final_model = lgb.train(
    final_params,
    train_data_full,
    num_boost_round=700,
    callbacks=[lgb.log_evaluation(100)]
)

print("✓ Production model fitted on entire data registry")

#  5-Fold Stratified Cross-Validation Evaluation 
print("\n" + "=" * 70)
print("5-Fold Cross-Validation Evaluation Framework")
print("=" * 70)

cv_scores_final = []
cv_accuracies = []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_fold_train = X_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]
    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]
    
    train_fold_data = lgb.Dataset(X_fold_train, label=y_fold_train)
    val_fold_data = lgb.Dataset(X_fold_val, label=y_fold_val, reference=train_fold_data)
    
    fold_model = lgb.train(
        final_params,
        train_fold_data,
        num_boost_round=700,
        valid_sets=[val_fold_data],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
    )
    
    y_pred_proba = fold_model.predict(X_fold_val)
    y_pred = (y_pred_proba > optimal_threshold).astype(int)
    
    accuracy = accuracy_score(y_fold_val, y_pred)
    roc_auc = roc_auc_score(y_fold_val, y_pred_proba)
    
    cv_scores_final.append(roc_auc)
    cv_accuracies.append(accuracy)
    
    print(f"Fold {fold+1}: Accuracy Evaluation={accuracy:.6f}, ROC-AUC Performance Metric={roc_auc:.6f}")

print(f"\n✓ Consolidated Global Mean Accuracy: {np.mean(cv_accuracies):.6f} (+/- {np.std(cv_accuracies):.6f})")
print(f"\n✓ Consolidated Global Mean ROC-AUC: {np.mean(cv_scores_final):.6f} (+/- {np.std(cv_scores_final):.6f})")

#  Empirical Feature Importance Analysis 
print("\n" + "=" * 70)
print("Empirical Feature Importance Diagnostics")
print("=" * 70)

feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': final_model.feature_importance()
}).sort_values('importance', ascending=False)

print("\nTop 20 Driver Predictive Features Profile:")
print(feature_importance.head(20).to_string(index=False))

# Visualization Routine
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 8))
plt.barh(feature_importance['feature'][:20], feature_importance['importance'][:20])
plt.xlabel('Split Frequency Importance Count')
plt.title('LightGBM Predictive Feature Importance Metrics (Top 20)')
plt.tight_layout()
plt.savefig('feature_importance_v2.png', dpi=100, bbox_inches='tight')
plt.close()

print("\n✓ Feature importance plot generated and exported to storage registry: feature_importance_optimized.png")


Final Full Model Training Production Run
✓ Production model fitted on entire data registry

5-Fold Cross-Validation Evaluation Framework
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[132]	valid_0's binary_logloss: 0.371664
Fold 1: Accuracy Evaluation=0.815411, ROC-AUC Performance Metric=0.910009
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[125]	valid_0's binary_logloss: 0.389941
Fold 2: Accuracy Evaluation=0.803335, ROC-AUC Performance Metric=0.900309
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[138]	valid_0's binary_logloss: 0.382125
Fold 3: Accuracy Evaluation=0.809661, ROC-AUC Performance Metric=0.903596
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[101]	valid_0's binary_logloss: 0.37707
Fold 4: Accuracy Evaluation=0.803797, ROC-AUC Performance Metric=0.907566
Training until vali

In [6]:
print("\n" + "=" * 70)
print("Unseen Test Registry Target Inference Engine")
print("=" * 70)

# Pass test instances through full production model using optimized operational threshold
y_test_proba = final_model.predict(X_test)
y_test_pred = (y_test_proba > optimal_threshold).astype(int)

print(f"✓ Inference run complete on evaluation registry")
print(f"  Distribution Ratio Classified as Transported (True): {y_test_pred.mean():.4f}")
print(f"  Distribution Ratio Classified as Non-Transported (False): {(1-y_test_pred.mean()):.4f}")

#  Create Submission Delivery File 
submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Transported': y_test_pred.astype(bool)
})

print("\nSubmission Package File Preview Structure:")
print(submission.head(10))

# Export submission manifest
submission.to_csv('submission_lightgbm_v2.csv', index=False)

print(f"\n✓ Verified submission manifest exported successfully: submission_lightgbm_optimized.csv")
print(f"  Total Manifest Row Records: {len(submission)}")
print(f"  Target Instance Counts Classified as Transported (True): {submission['Transported'].sum()}")
print(f"  Target Instance Counts Classified as Non-Transported (False): {(~submission['Transported']).sum()}")

print("\n" + "=" * 70)
print("=" * 70)


Unseen Test Registry Target Inference Engine
✓ Inference run complete on evaluation registry
  Distribution Ratio Classified as Transported (True): 0.5478
  Distribution Ratio Classified as Non-Transported (False): 0.4522

Submission Package File Preview Structure:
  PassengerId  Transported
0     0013_01         True
1     0018_01        False
2     0019_01         True
3     0021_01         True
4     0023_01         True
5     0027_01        False
6     0029_01         True
7     0032_01         True
8     0032_02         True
9     0033_01         True

✓ Verified submission manifest exported successfully: submission_lightgbm_optimized.csv
  Total Manifest Row Records: 4277
  Target Instance Counts Classified as Transported (True): 2343
  Target Instance Counts Classified as Non-Transported (False): 1934

